<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_CHARGING_STATION_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import math
import random
from concurrent.futures import ThreadPoolExecutor

# Global parameters
AVG_SPEED = 40.0       # km/h
AVG_CHARGE_TIME = 0.5  # hours (30 mins)
SWITCH_PROB = 0.7      # Probability to switch to prevent oscillation

def calculate_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def evaluate_cost(ev_idx, station_idx, ev_coords, stations, station_loads):
    dist = calculate_distance(ev_coords[ev_idx], stations[station_idx]['coords'])
    travel_time = dist / AVG_SPEED

    # Calculate queue waiting time based on load and number of chargers
    load = station_loads[station_idx]
    capacity = stations[station_idx]['capacity']

    if load <= capacity:
        wait_time = 0.0
    else:
        # Estimated wait time for excess vehicles
        wait_time = ((load - capacity) / capacity) * AVG_CHARGE_TIME

    return travel_time + wait_time

def update_cell(args):
    ev_idx, current_assignments, ev_coords, stations, station_loads = args
    current_station = current_assignments[ev_idx]

    # Cost at current station
    current_cost = evaluate_cost(ev_idx, current_station, ev_coords, stations, station_loads)

    best_station = current_station
    min_cost = current_cost

    # Check neighboring/all stations
    for s_idx in range(len(stations)):
        if s_idx == current_station:
            continue
        # Simulate load if this EV moves to s_idx
        simulated_loads = station_loads.copy()
        simulated_loads[s_idx] += 1

        cost = evaluate_cost(ev_idx, s_idx, ev_coords, stations, simulated_loads)
        if cost < min_cost:
            min_cost = cost
            best_station = s_idx

    # Stochastic update to prevent all EVs rushing to the same station simultaneously
    if best_station != current_station and random.random() < SWITCH_PROB:
        return best_station
    return current_station

def parallel_cellular_optimization(ev_coords, stations, max_iters=30):
    num_evs = len(ev_coords)
    num_stations = len(stations)

    # Step 1: Random initial assignment
    assignments = [random.randint(0, num_stations - 1) for _ in range(num_evs)]

    # Parallel execution across CPU threads
    with ThreadPoolExecutor() as executor:
        for iteration in range(max_iters):
            # Calculate current loads per station
            station_loads = [0] * num_stations
            for s_idx in assignments:
                station_loads[s_idx] += 1

            # Prepare arguments for parallel cell updates
            tasks = [(i, assignments, ev_coords, stations, station_loads) for i in range(num_evs)]

            # Step 2 & 3: Synchronous parallel update
            new_assignments = list(executor.map(update_cell, tasks))

            if new_assignments == assignments:
                break
            assignments = new_assignments

    return assignments

def main():
    print("=== EV Charging Station Allocation (Parallel Cellular Algorithm) ===\n")

    # User Input for Stations
    num_stations = int(input("Enter number of charging stations: "))
    stations = []
    for i in range(num_stations):
        print(f"\n--- Station {i+1} ---")
        x = float(input(f"Enter X coordinate for Station {i+1}: "))
        y = float(input(f"Enter Y coordinate for Station {i+1}: "))
        cap = int(input(f"Enter number of chargers (capacity) at Station {i+1}: "))
        stations.append({'id': i+1, 'coords': (x, y), 'capacity': cap})

    # User Input for EVs
    num_evs = int(input("\nEnter number of Electric Vehicles (EVs): "))
    ev_coords = []
    for i in range(num_evs):
        print(f"\n--- EV {i+1} ---")
        x = float(input(f"Enter X coordinate for EV {i+1}: "))
        y = float(input(f"Enter Y coordinate for EV {i+1}: "))
        ev_coords.append((x, y))

    iterations = int(input("\nEnter maximum algorithm iterations (e.g., 30): "))

    # Run Algorithm
    optimal_assignments = parallel_cellular_optimization(ev_coords, stations, iterations)

    # Calculate final metrics
    final_loads = [0] * num_stations
    for s_idx in optimal_assignments:
        final_loads[s_idx] += 1

    print("\n" + "="*50)
    print("OPTIMAL EV CHARGING ALLOCATION RESULTS")
    print("="*50)

    total_time = 0.0
    for i in range(num_evs):
        assigned_s = optimal_assignments[i]
        cost_hrs = evaluate_cost(i, assigned_s, ev_coords, stations, final_loads)
        cost_mins = cost_hrs * 60
        total_time += cost_mins
        dist = calculate_distance(ev_coords[i], stations[assigned_s]['coords'])
        print(f"EV {i+1} at {ev_coords[i]}  -->  Assigned to Station {assigned_s+1} "
              f"(Dist: {dist:.2f} km | Est. Total Time: {cost_mins:.1f} mins)")

    print("\n--- Station Utilization Summary ---")
    for j in range(num_stations):
        cap = stations[j]['capacity']
        load = final_loads[j]
        util = (load / cap) * 100
        print(f"Station {j+1} at {stations[j]['coords']}: {load} EVs assigned / {cap} Chargers "
              f"(Load Ratio: {util:.1f}%)")

    print(f"\nAverage Time per EV (Travel + Wait): {total_time / num_evs:.2f} minutes")
    print("="*50)

if __name__ == "__main__":
    main()
1

=== EV Charging Station Allocation (Parallel Cellular Algorithm) ===

Enter number of charging stations: 2

--- Station 1 ---
Enter X coordinate for Station 1: 0
Enter Y coordinate for Station 1: 0
Enter number of chargers (capacity) at Station 1: 2

--- Station 2 ---
Enter X coordinate for Station 2: 10
Enter Y coordinate for Station 2: 0
Enter number of chargers (capacity) at Station 2: 3

Enter number of Electric Vehicles (EVs): 4

--- EV 1 ---
Enter X coordinate for EV 1: 1
Enter Y coordinate for EV 1: 2

--- EV 2 ---
Enter X coordinate for EV 2: 5
Enter Y coordinate for EV 2: 0

--- EV 3 ---
Enter X coordinate for EV 3: 3
Enter Y coordinate for EV 3: 6

--- EV 4 ---
Enter X coordinate for EV 4: 2
Enter Y coordinate for EV 4: 5

Enter maximum algorithm iterations (e.g., 30): 25

OPTIMAL EV CHARGING ALLOCATION RESULTS
EV 1 at (1.0, 2.0)  -->  Assigned to Station 2 (Dist: 9.22 km | Est. Total Time: 13.8 mins)
EV 2 at (5.0, 0.0)  -->  Assigned to Station 1 (Dist: 5.00 km | Est. Total 